# 02 — 离散隐变量：直接 MLE 与 EM
沿用第 01 章的二维数据与 MLE 目标，只改变概率模型：不再直接假设观测来自单个高斯，而是假设先抽取类别 $x\in\{0,1,2\}$，再从该类别的高斯分布生成观测。

主动定义 $p(x=k)=\pi_k$ 与 $p(y|x=k)=\mathcal N(y;\mu_k,\sigma^2I)$。固定 $\sigma=0.45$；待学习参数是类别概率 $\pi$ 与三个二维均值 $\mu_k$。类别标签不可观测。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, multivariate_normal
from utils import generate_observations, compare, plot_history

# 共享数据，只观测 y；不读取真实生成类别。
rng = np.random.default_rng(42)
y = generate_observations(rng)

## 1. 给定初始参数，计算五个概率量
先不训练。定义先验和条件分布，乘法得到联合密度，对类别求和得到边缘密度，最后可选地计算后验。后验用于隐变量推断。

In [ ]:
# 固定观测噪声；mu0、pi0 是人为选择的初始参数，不是真值。
sigma = .45
mu0 = np.array([[-1., 0.], [1., 0.], [0., 2.]])
pi0 = np.ones(3) / 3

def probabilities(y, mu, pi):
    N = len(y)
    K = len(pi)
    # prior[k] = p(x=k)，由当前参数直接确定。
    prior = pi.copy()
    conditional = np.zeros((N, K))
    joint = np.zeros((N, K))
    marginal = np.zeros(N)
    posterior = np.zeros((N, K))

    # 第 k 个条件分布 p(y|x=k)，由当前均值和固定协方差定义。
    models = []
    for k in range(K):
        models.append(multivariate_normal(mu[k], sigma**2 * np.eye(2)))

    # 对每个观测样本，逐个枚举隐变量的取值。
    for i in range(N):
        for k in range(K):
            conditional[i, k] = models[k].pdf(y[i])
            # p(y_i,x=k) = p(y_i|x=k) p(x=k)
            joint[i, k] = conditional[i, k] * prior[k]

        # 对隐变量求和：p(y_i)
        for k in range(K):
            marginal[i] += joint[i, k]

        # 贝叶斯公式：p(x=k | y_i)
        for k in range(K):
            posterior[i, k] = joint[i, k] / marginal[i]

    return prior, conditional, joint, marginal, posterior

parts = probabilities(y, mu0, pi0)
prior, conditional, joint, marginal, posterior = parts
print('p(x=k), k=0,1,2:', prior)
print('density p(y_0|x=k):', conditional[0])
print('joint density p(y_0,x=k):', joint[0])
print('density p(y_0):', marginal[0])
print('probability p(x=k|y_0):', posterior[0])

for i in range(len(y)):
    assert np.isclose(sum(parts[-1][i]), 1)

## 2. 沿用第 01 章的最大似然目标
仍然最大化 $\sum_i\log p_\theta(y_i)$，但这里 $p_\theta(y_i)=\sum_k p_\theta(y_i,x=k)$。不同分量出现在 log 内部的求和中，一般不能像第 01 章那样直接求出解析参数解。改用 PyTorch 梯度优化。

logits 是三个无约束的实数，经 softmax 得到非负且和为 1 的类别概率。logsumexp 计算 $\log\sum_k\exp(a_k)$，与先求和再取 log 等价，但数值更稳定。

训练时，每个类别的 MultivariateNormal.log_prob(Y) 返回 N 个二维观测的对数密度。stack 将 K 个长度 N 的向量组成 [N,K]；dim=1 是类别方向，不是数据方向。后验不参与直接 MLE。

In [ ]:
import torch
torch.set_num_threads(1)
torch.manual_seed(42)
Y = torch.tensor(y, dtype=torch.float64)
mu = torch.nn.Parameter(torch.tensor(mu0.copy()))
logits = torch.nn.Parameter(torch.zeros(3, dtype=torch.float64))
R = sigma**2 * torch.eye(2, dtype=Y.dtype)
opt = torch.optim.Adam([mu, logits], lr=.05)

def log_marginal(Y, mu, logits):
    # 定义先验：log_pi[k] = log p(x=k)
    log_pi = logits.log_softmax(dim=0)
    # 每一列保存一个类别对应的所有样本联合对数密度。
    columns = []
    for k in range(len(pi0)):
        conditional = torch.distributions.MultivariateNormal(mu[k], R)
        log_conditional = conditional.log_prob(Y)
        log_joint = log_conditional + log_pi[k]
        columns.append(log_joint)
    # 形状为 [样本数, 类别数]；每行对应一个观测。
    log_joint_values = torch.stack(columns, dim=1)
    # 沿类别方向求和（在 log 空间实现）；返回每个样本的 log p(y_i)。
    return torch.logsumexp(log_joint_values, dim=1)

mle_history = []
for epoch in range(150):
    # 最大化平均观测对数似然。
    ll = log_marginal(Y, mu, logits).mean()
    loss = -ll
    opt.zero_grad()
    loss.backward()
    opt.step()
    mle_history.append(ll.item())

mu_mle = mu.detach().numpy()
pi_mle = logits.softmax(dim=0).detach().numpy()
print('means:', mu_mle)
print('prior:', pi_mle)
plot_history(mle_history)

## 3. 从训练完成的模型生成数据
先从学习到的类别先验抽取 $x$，再从对应条件分布抽取 $y$。这里不需要后验。

In [ ]:
samples = []
for i in range(len(y)):
    k = rng.choice(len(pi_mle), p=pi_mle)
    conditional = multivariate_normal(mu_mle[k], sigma**2*np.eye(2))
    samples.append(conditional.rvs(random_state=rng))
compare(y, np.array(samples))

## 4. 同一个 MLE 问题：EM 算法

记 $\theta=(\pi,\mu)$，$q_i(k)$ 为第 $i$ 个观测对应的辅助类别分布。

$$
\log p_\theta(y_i)=\mathcal L_i(q_i,\theta)+D_{\mathrm{KL}}(q_i\|p_\theta(x|y_i))
$$

其中

$$
\mathcal L_i(q_i,\theta)=\sum_k q_i(k)\log\frac{p_\theta(y_i,x=k)}{q_i(k)}.
$$

E step：固定旧参数 $\theta^t$，利用第 1 节的贝叶斯公式计算 $q_i(k)=p_{\theta^t}(x=k|y_i)$，使 KL 为零。

M step：固定所有 $q_i$，最大化 ELBO。此时只有 $\sum_{i,k}q_i(k)\log p_\theta(y_i,x=k)$ 依赖参数。注意 KL 随新参数变化，并没有被当作常数删除。

利用当前模型的联合密度展开：

$$
Q(\theta,\theta^t)=\sum_{i,k}q_i(k)\left[\log\pi_k-\frac{\|y_i-\mu_k\|^2}{2\sigma^2}\right]+C.
$$

对均值求导，对类别概率在 $\sum_k\pi_k=1$ 的约束下求极值，得到解析 M step：

$$
N_k=\sum_i q_i(k),\qquad \pi_k^{t+1}=\frac{N_k}{N},\qquad
\mu_k^{t+1}=\frac{\sum_i q_i(k)y_i}{N_k}.
$$

单调性来自旧位置的等号、M step 提高下界、KL 非负：

$$
\ell(\theta^t)=\sum_i\mathcal L_i(q_i,\theta^t)
\leq\sum_i\mathcal L_i(q_i,\theta^{t+1})
\leq\ell(\theta^{t+1}).
$$

这里 $\ell(\theta)=\sum_i\log p_\theta(y_i)$。EM 和直接梯度法使用相同数据、模型、初始参数与最终目标，只是优化方法不同。

In [ ]:
# 使用与直接 MLE 完全相同的初始参数。
mu_em = mu0.copy()
pi_em = pi0.copy()
N, K = len(y), len(pi0)

def mean_log_likelihood(mu_value, pi_value):
    with torch.no_grad():
        return log_marginal(
            Y, torch.tensor(mu_value), torch.tensor(np.log(pi_value))
        ).mean().item()

def mean_elbo(q, joint):
    # q 和 joint 均为 [N, K]；q 为零的项按极限取零。
    value = 0.
    for i in range(N):
        for k in range(K):
            if q[i, k] > 0:
                value += q[i, k] * (np.log(joint[i, k]) - np.log(q[i, k]))
    return value / N

em_history = [mean_log_likelihood(mu_em, pi_em)]
for epoch in range(30):
    # E step：复用第 1 节的后验计算，不更新模型参数。
    _, _, joint_old, _, q = probabilities(y, mu_em, pi_em)
    elbo_old = mean_elbo(q, joint_old)

    # M step：固定 q，逐样本累加每个类别的权重及加权观测。
    for k in range(K):
        N_k = 0.
        weighted_y = np.zeros(y.shape[1])
        for i in range(N):
            N_k += q[i, k]
            weighted_y += q[i, k] * y[i]
        pi_em[k] = N_k / N
        mu_em[k] = weighted_y / N_k

    # 用旧 q 和新参数计算下界，并验证单调性证明的三个关系。
    _, _, joint_new, _, _ = probabilities(y, mu_em, pi_em)
    elbo_new = mean_elbo(q, joint_new)
    ll_new = mean_log_likelihood(mu_em, pi_em)
    assert np.isclose(elbo_old, em_history[-1])
    assert elbo_new >= elbo_old - 1e-9
    assert ll_new >= elbo_new - 1e-9
    em_history.append(ll_new)
    if abs(ll_new - em_history[-2]) < 1e-9:
        break

print('EM means:', mu_em)
print('EM prior:', pi_em)
print('EM mean log likelihood:', em_history[-1])
print('Direct MLE mean log likelihood:', mean_log_likelihood(mu_mle, pi_mle))
plot_history(em_history)

两条曲线均为真实观测平均对数似然，而非 ELBO。横轴分别是梯度更新次数和 EM 迭代次数，不能直接解释为运行时间。

In [ ]:
# 梯度法的历史记录在更新前，补上最后一次更新后的目标值。
plt.plot(mle_history + [mean_log_likelihood(mu_mle, pi_mle)], label='Direct MLE (Adam)')
plt.plot(em_history, label='EM')
plt.xlabel('Iteration')
plt.ylabel('Mean log likelihood')
plt.legend()
plt.show()

## 小结
引入离散隐变量并未改变 MLE 目标。直接梯度法优化边缘似然；EM 交替计算精确后验与最大化固定后验下的 ELBO，本例的 M step 有解析解。下一章将隐变量从离散类别改为连续标量，对隐变量求和将变成积分。